## 1. Importar librerías

In [14]:
import os
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
import joblib

## 2. Cargar datasets

In [15]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/validation.csv")
test = pd.read_csv("../data/processed/test.csv")

## 3. Separar features y target

In [16]:
TARGET = "helada_24h"

X_train = train.drop(columns=["fecha_hora", TARGET])
y_train = train[TARGET]

X_val = val.drop(columns=["fecha_hora", TARGET])
y_val = val[TARGET]

X_test = test.drop(columns=["fecha_hora", TARGET])
y_test = test[TARGET]

## 4. Crear modelos

### Árbol de decisión

In [17]:
dt_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=10,
    class_weight="balanced",
    random_state=42
)

### Random Forest

In [18]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

### SVM (con pipeline)

In [19]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

### MLP (Red Neuronal)

In [20]:
mlp_model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation="relu",
        max_iter=300,
        alpha=0.001,
        learning_rate_init=0.001,
        random_state=42
    ))
])

## 5. Función de Entrenamiento + Evaluación

In [21]:
def train_and_evaluate(name, model, X_train, y_train, X_val, y_val, X_test, y_test):
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    results = {
        "modelo": name,

        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_precision": precision_score(y_val, val_pred),
        "val_recall": recall_score(y_val, val_pred),
        "val_f1": f1_score(y_val, val_pred),

        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_precision": precision_score(y_test, test_pred),
        "test_recall": recall_score(y_test, test_pred),
        "test_f1": f1_score(y_test, test_pred),
    }

    return model, results

## 6. Entrenar todos los modelos

In [22]:
results_list = []

dt_model, dt_res = train_and_evaluate(
    "Decision Tree", dt_model,
    X_train, y_train, X_val, y_val, X_test, y_test
)

rf_model, rf_res = train_and_evaluate(
    "Random Forest", rf_model,
    X_train, y_train, X_val, y_val, X_test, y_test
)

svm_model, svm_res = train_and_evaluate(
    "SVM", svm_model,
    X_train, y_train, X_val, y_val, X_test, y_test
)

mlp_model, mlp_res = train_and_evaluate(
    "MLP", mlp_model,
    X_train, y_train, X_val, y_val, X_test, y_test
)

results_list.extend([dt_res, rf_res, svm_res, mlp_res])

d:\Descargas\2026-1\Inteligencia Artificial\Lab 5 2025-2\lab_mlp\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## 7. Tabla final de métricas

In [23]:
results_df = pd.DataFrame(results_list)

results_df

,modelo,val_accuracy,val_precision,val_recall,val_f1,test_accuracy,test_precision,test_recall,test_f1
0,Decision Tree,0.905392,0.770706,0.982222,0.863703,0.737627,0.355852,0.794055,0.491459
1,Random Forest,0.916921,0.801843,0.966667,0.876574,0.793559,0.434160,0.966030,0.599078
2,SVM,0.814174,0.809859,0.511111,0.626703,0.821356,0.433962,0.390658,0.411173
3,MLP,0.759580,0.920705,0.232222,0.370896,0.826441,0.443213,0.339703,0.384615


## 8. Guardar modelos

In [24]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

joblib.dump(dt_model, "../models/decision_tree.pkl")
joblib.dump(rf_model, "../models/random_forest.pkl")
joblib.dump(svm_model, "../models/svm.pkl")
joblib.dump(mlp_model, "../models/neural_network.pkl")

['../models/neural_network.pkl']

## 9. Guardar métricas

In [25]:
results_df.to_csv("../results/metricas_modelos.csv", index=False)

## 10. Modelo Ganador

In [26]:
best_model = results_df.sort_values("test_f1", ascending=False)
best_model

,modelo,val_accuracy,val_precision,val_recall,val_f1,test_accuracy,test_precision,test_recall,test_f1
1,Random Forest,0.916921,0.801843,0.966667,0.876574,0.793559,0.434160,0.966030,0.599078
0,Decision Tree,0.905392,0.770706,0.982222,0.863703,0.737627,0.355852,0.794055,0.491459
2,SVM,0.814174,0.809859,0.511111,0.626703,0.821356,0.433962,0.390658,0.411173
3,MLP,0.759580,0.920705,0.232222,0.370896,0.826441,0.443213,0.339703,0.384615
